# 07 — Teacher–Student Online Co-Training

Train two MOUSE models together on live `mouse-gym` rollouts:

- **Student** (`Qwen3Backbone`, causal): explores the environment and collects replay. Head: `DiscreteActionHead` (action logits). Rollout uses KV cache.
- **Teacher** (`Qwen3Backbone`, causal): learns Q-values online from the same replay via `DqnObjective`. Head: `DiscreteActionValueHead`.

Each training step runs one forward on each model on the same batch. The teacher gets a Bellman TD loss; the student distills the teacher's Q into action logits via `SpObjective` (`targets_key="action_value"`). One shared AdamW optimizer updates both models.


In [ ]:
from collections import deque

import torch

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_env
from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionHead, DiscreteActionValueHead
from mouse_core.objectives import DqnObjective, SpObjective


STUDENT_MODEL_ID = "mouse-example-model-distill-online"
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
NUM_ENVS = 1000

GRADIENT_STEPS = 20000
SEQUENCE_LENGTH = 512
BATCH_SIZE = 4

ENV_STEPS_PER_CYCLE = 1000
STEPS_PER_ENV = 100
GRADIENT_STEPS_PER_CYCLE = 1000

LEARNING_STARTS = 2000
EXPLORATION_ENDS = 10000


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Online training keeps the environment in the training loop. Each `SingleEnv` instance runs one infinite task with deterministic Procedural Frozen Lake dynamics and 50-step episode truncation, so the model keeps carrying policy context across episode boundaries.

Rewards are unshaped: reaching the goal pays 1.0 and everything else pays 0. The pressure to make progress comes from the 50-step truncation — a policy that loops (e.g. bumps a wall forever) collects nothing before the episode ends.


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "max_episode_steps": 50,
            "map_seed": i,
        },
    )
    for i in range(NUM_ENVS)
]

envs = [make_env(config) for config in configs]
print(f"Environments {min(10, len(envs))} of {len(envs)}:")
for env in envs[:10]:
    print(env.name)

## Build teacher and student

Both use causal `Qwen3Backbone`. The **teacher** has a Q head (`DiscreteActionValueHead`); the **student** has an action-logits head (`DiscreteActionHead`) for `SpObjective` distillation.

Both share the same modality layout as `03_train_online.ipynb`. Truncate layers for faster debugging:

```python
# teacher_backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=4)
# student_backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=4)
```


In [ ]:
teacher_backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")
student_backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=14)

teacher_encoder = StepEmbedder(
    hidden_dim=teacher_backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

student_encoder = StepEmbedder(
    hidden_dim=student_backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

teacher_head = DiscreteActionValueHead(
    in_features=teacher_backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=teacher_backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

student_head = DiscreteActionHead(
    in_features=student_backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=student_backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

teacher = Model(
    encoder=teacher_encoder,
    backbone=teacher_backbone,
    heads=teacher_head,
).train().to(device)

student = Model(
    encoder=student_encoder,
    backbone=student_backbone,
    heads=student_head,
).train().to(device)

print("Teacher:")
print(teacher)
print("Student:")
print(student)


## Replay buffer and policy contexts

Each env writes to one `Datastore`; together they are the live replay buffer. Each env also has a `contexts` deque capped at `SEQUENCE_LENGTH` — the action-selection history used during collection. Contexts are never cleared; episode boundaries appear as `done` values in the sequence.


In [ ]:
stores = [Datastore(name=env.name) for env in envs]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in envs]


## Rollout phase

Only the **student** interacts with the env (ε-greedy on action logits, KV cache). The teacher is not called during collection. Envs are visited round-robin across rollouts via a persistent `env_cursor`, so every env contributes data over the course of training.


In [ ]:
def epsilon_for_env_step(env_step: int) -> float:
    """Linear ε decay: 1.0 (explore) → 0.0 (greedy)."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    return 1.0 - frac


In [ ]:
def rollout(
    model: Model,
    envs: list,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    grad_steps: int,
    env_cursor: int,
) -> tuple[int, int]:
    """Collect up to ``ENV_STEPS_PER_CYCLE`` env transitions, then return.

    Envs are visited round-robin starting at ``env_cursor``; the updated cursor
    is returned so the next rollout continues where this one stopped.
    """
    for env in envs:
        env.metrics.clear()
    model.eval()

    budget = ENV_STEPS_PER_CYCLE
    collected = 0

    for _ in range(len(envs)):
        if collected >= budget:
            break
        env = envs[env_cursor]
        store = stores[env_cursor]
        context = contexts[env_cursor]
        env_cursor = (env_cursor + 1) % len(envs)

        kv_cache = None
        cache_count = 0
        ctx_count = 0
        chunk = min(STEPS_PER_ENV, budget - collected)

        for _ in range(chunk):
            epsilon = epsilon_for_env_step(env_steps)
            if not context or torch.rand(1).item() < epsilon:
                inp = env.sample_random_input()
            else:
                ctx_list = list(context)
                with torch.no_grad():
                    if kv_cache is None:
                        preds, _, kv_cache = model([ctx_list], use_cache=True)
                    else:
                        uncached = ctx_count - cache_count
                        preds, _, kv_cache = model(
                            [ctx_list[-uncached:]], cache=kv_cache, use_cache=True
                        )
                    cache_count = ctx_count

                action = model.get_action(preds, temperature=0.0, num_actions=MAX_ACTIONS)
                inp = {"action": action.squeeze().cpu().numpy()}

            out = env.step(inp)
            row = {**inp, **out}
            row.pop("info", None)  # keep the replay schema flat
            store.append(row)
            context.append(row)
            ctx_count += 1
            env_steps += 1
            collected += 1

    rewards: list[float] = []
    lengths: list[int] = []
    for env in envs:
        rewards.extend(env.metrics.episode_cum_rewards)
        lengths.extend(env.metrics.episode_lengths)
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).float().mean().item() if n else float("nan")
    epsilon = epsilon_for_env_step(env_steps)
    print(
        f"env_step={env_steps} grad_step={grad_steps} epsilon={epsilon:.3f}"
        f" | {n} completed episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}"
    )
    return env_steps, env_cursor


## Training phase

Each optimizer step:

1. Forward **student** and **teacher** on the same replay batch.
2. `DqnObjective` on teacher Q (`action_value` / `action_value_target`).
3. Inject detached teacher online Q into `objective_data["action_value"]`; `SpObjective` (`targets_key="action_value"`) on student logits (`action`).
4. Sum losses, one backward, one optimizer step; polyak-update teacher target net.

Both models are causal: Q targets at each step depend only on the prefix, matching what the student sees at rollout time.


In [ ]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)

dqn_objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=0.99,
    gamma_episode_truncated=0.99,
    gamma_task_terminal=0.99,
    gamma_task_truncated=0.99,
    tau=0.0005,
)

sp_objective = SpObjective(
    loss_type="js",
    temperature=1.0,
    targets_key="action_value",
    predictions_key="action",
)

optimizer = torch.optim.AdamW(
    list(student.parameters()) + list(teacher.parameters()),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)


In [ ]:
def train_burst(
    student: Model,
    teacher: Model,
    optimizer: torch.optim.Optimizer,
    dqn_objective: DqnObjective,
    sp_objective: SpObjective,
    loader: DataLoader,
    env_steps: int,
    grad_steps: int,
    burst_steps: int,
) -> tuple[torch.Tensor, dict[str, float], dict[str, float]]:
    """Run ``burst_steps`` co-training updates on shared replay batches."""
    student.train()
    teacher.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    teacher_metrics: dict[str, float] = {}
    student_metrics: dict[str, float] = {}
    for _ in range(burst_steps):
        batch = loader.next_batch()

        student_preds, objective_data, _ = student(batch)
        teacher_preds, _, _ = teacher(batch)

        teacher_loss, teacher_metrics = dqn_objective(objective_data, teacher_preds)

        distill_data = objective_data.update(
            {"action_value": teacher_preds["action_value"].detach()}
        )
        student_loss, student_metrics = sp_objective(distill_data, student_preds)

        loss = teacher_loss + student_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        teacher.polyak_update(action_value_tau=dqn_objective.tau)

    assert loss is not None
    print(
        f"env_step={env_steps} grad_step={grad_steps + burst_steps}"
        f" | loss={loss.item():.4f} teacher_q={teacher_metrics['q_values_mean']:.3f}"
        f" sp={student_metrics['action']:.4f}"
    )
    return loss, teacher_metrics, student_metrics


## Run online co-training

The master loop alternates **student rollout → co-train burst** until `GRADIENT_STEPS` optimizer updates have been completed.


In [ ]:
env_steps = 0
grad_steps = 0
env_cursor = 0

while grad_steps < GRADIENT_STEPS:
    env_steps, env_cursor = rollout(
        student, envs, stores, contexts, env_steps, grad_steps, env_cursor
    )

    if env_steps >= LEARNING_STARTS:
        burst = min(GRADIENT_STEPS_PER_CYCLE, GRADIENT_STEPS - grad_steps)
        train_burst(
            student,
            teacher,
            optimizer,
            dqn_objective,
            sp_objective,
            loader,
            env_steps,
            grad_steps,
            burst,
        )
        grad_steps += burst

loader.close()

for env in envs:
    env.close()
print(f"Co-training finished ({grad_steps} optimizer steps, {env_steps} env steps).")


## Push student to the Hub

Deploy the causal **student** (action logits + KV cache). Inference follows `examples/04_inference.ipynb` with `STUDENT_MODEL_ID`.


In [ ]:
student.eval().to("cpu")
url = push_model_to_hub(student, STUDENT_MODEL_ID, private=False, clear=True)
print(f"Pushed student to {url}")
